In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction = 16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(channels, channels//reduction),
            nn.ReLU(),
            nn.Linear(channels//reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        scale = self.se(inputs)
        scale = scale.view(scale.shape[0], -1, 1, 1)
        return inputs * scale

In [ ]:
class SEResidualUnit(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.main_layers = nn.Sequential(
            nn.Conv2d(in_channels=input_channels, out_channels=output_channels, kernel_size=3, stride=stride, padding=1 if stride > 1 else "same"),
            nn.BatchNorm2d(num_features=output_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels=output_channels, out_channels=output_channels, kernel_size=3, stride=1, padding="same"),
            nn.BatchNorm2d(num_features=output_channels),
        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                nn.Conv2d(in_channels=input_channels, out_channels=output_channels, kernel_size=1, stride=stride, padding=0),
                nn.BatchNorm2d(num_features=output_channels),
            )
        else:
            self.skip_connection = nn.Identity()
        self.se_block = SEBlock(output_channels)

    def forward(self, inputs):
        x = self.main_layers(inputs) + self.skip_connection(inputs)
        x = self.se_block(x)
        return F.relu(x)

In [ ]:
class SENet(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2,
                      padding=3, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(SEResidualUnit(prev_filters, filters, stride=stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10),
        ]
        self.resnet = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.resnet(inputs)